# Prepare Stage 2 Vietnam Classifier Dataset

Notebook này chỉ thực hiện **extract, audit, report** cho Dataset2 Việt Nam v19.

- Giữ nguyên split `train/valid/test`.
- Chỉ dùng class `rider_with_helmet` và `rider_without_helmet`.
- Crop upper 50% rider + padding; không augmentation và không resize source.
- Không import detector/classifier và không train model.
- Dừng nếu không tìm thấy metadata chính xác hoặc có exact leakage giữa split.

In [ ]:
from pathlib import Path
import sys

# Local: search from the current project. Colab: search Drive first, then /content.
search_roots = [Path.cwd()]
if Path('/content').exists():
    search_roots = []
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        search_roots.append(drive_root)
    search_roots.append(Path('/content'))

for module_root in search_roots + [Path.cwd(), Path('/content/drive/MyDrive')]:
    module_path = module_root / 'prepare_stage2_vietnam_classifier.py'
    if module_path.exists() and str(module_root) not in sys.path:
        sys.path.insert(0, str(module_root))

from prepare_stage2_vietnam_classifier import DatasetError, find_vietnam_dataset, run_preparation

print('Search roots:')
for root in search_roots:
    print(' -', root.resolve())

In [ ]:
# Preflight: identify exactly one valid Dataset2 Vietnam v19 source.
source_root = None
metadata = None
split_dirs = None
errors = []
for root in search_roots:
    try:
        metadata, split_dirs = find_vietnam_dataset(root)
        source_root = root
        break
    except DatasetError as exc:
        errors.append(f'{root}: {exc}')

if source_root is None:
    raise DatasetError('STOP: Dataset2 Vietnam v19 was not found in the configured local/Colab roots.\n' + '\n'.join(errors))

print('Dataset:', metadata.yaml_path)
print('Workspace/project/version:', metadata.workspace, metadata.project, metadata.version)
for split, path in split_dirs.items():
    print(f'{split}: {path}')

In [ ]:
# Run extraction and all audits. Test is created but never used for tuning.
output_root = source_root / 'Stage2_Vietnam'
run_preparation(source_root, output_root=output_root, seed=42)

Sau khi chạy thành công, xem `Stage2_Vietnam/manifest.csv`, `stage2_summary.txt` và các report/contact sheet. Nếu có leakage, cell chạy sẽ dừng với trạng thái STOP và không in READY.